# Spatial verification run — clean `repro` package (branch `rebuttal`)
**Upload-and-run on a GPU runtime.** One implementation, no vendored `src`;
every knob in `repro/configs/spatial.yaml`; every trainer seeds BEFORE
model construction.

Runs the full published protocol: 3 scenes x 5 seeds x 12-theta sweep,
12 detectors (DART, DART-CFAR, DARTS, DARTS-CFAR, AMF-global, AMF-local,
GMM-Levin, LRao(val-ES) + THANTD, HTD-Net, TSTTD, OS-VAE). Saves raw
scores per (seed, theta), per-seed metrics (per-class Pfa on Pavia),
checkpoints (LRao every 10 epochs), then **verifies against the published
numbers** and zips everything.

In [ ]:
!git clone -b rebuttal --depth 1 https://github.com/michaelpiro/final-paper-experiment.git repo
%cd repo
import os, sys, torch
sys.path.insert(0, '.')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
for p in ('repro/data/pavia-u.mat', 'repro/data/Sandiego.mat',
          'repro/data/Sandiego2.mat', 'repro/data/sandiego_regions.json',
          'repro/data/sandiego2_regions.json'):
    assert os.path.exists(p), f'missing {p}'
print('all bundled data present')

In [ ]:
from repro.protocols import spatial as SP
CFG = SP.load_cfg()
print('scenes:', CFG['scenes'], ' seeds:', CFG['seeds'])
print('thetas:', CFG['thetas'])

In [ ]:
SP.run_scene('pavia4', CFG, out_root='results/spatial', device=DEVICE)

In [ ]:
SP.run_scene('sandiego', CFG, out_root='results/spatial', device=DEVICE)

In [ ]:
SP.run_scene('sandiego2', CFG, out_root='results/spatial', device=DEVICE)

In [ ]:
SP.summarize('results/spatial')

In [ ]:
from repro.analysis.verify import verify_spatial
from repro.analysis.tables import make_tables
verify_spatial('results/spatial')
make_tables('results/spatial', dst='results/spatial/tables')

In [ ]:
import zipfile, os
with zipfile.ZipFile('spatial_verification.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk('results/spatial'):
        for fn in files:
            z.write(os.path.join(root, fn))
print('zipped -> spatial_verification.zip')
try:
    from google.colab import files
    files.download('spatial_verification.zip')
except Exception as e:
    print('manual download:', e)